<a href="https://colab.research.google.com/github/enslern/The-AI-engineer-course-2026-Complete-AI-bootcamp/blob/main/Exercise_2_POS_and_NER_tagging.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
nltk.download("wordnet")
from nltk.stem import WordNetLemmatizer
import spacy
import matplotlib.pyplot as plt
import pandas as pd
import re

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


CLEAN DATA

In [3]:
bbc_data=pd.read_csv("/content/drive/MyDrive/Colab Notebooks/resources/bbc_news.csv")

In [4]:
bbc_data.head()

,Unnamed: 0,index,title,pubDate,guid,link,description
0,0,6684,Can I refuse to work?,"Wed, 10 Aug 2022 15:46:18 GMT",https://www.bbc.co.uk/news/business-62147992,https://www.bbc.co.uk/news/business-62147992?a...,With much of the UK enduring another period of...
1,1,9267,'Liz Truss the Brief?' World reacts to UK poli...,"Mon, 17 Oct 2022 11:35:12 GMT",https://www.bbc.co.uk/news/world-63285480,https://www.bbc.co.uk/news/world-63285480?at_m...,The UK's political chaos has been watched arou...
2,2,7387,Rationing energy is nothing new for off-grid c...,"Wed, 31 Aug 2022 05:20:18 GMT",https://www.bbc.co.uk/news/uk-scotland-highlan...,https://www.bbc.co.uk/news/uk-scotland-highlan...,Scoraig in the north west Highlands has long h...
3,3,767,The hunt for superyachts of sanctioned Russian...,"Tue, 22 Mar 2022 14:37:01 GMT",https://www.bbc.co.uk/news/60739336,https://www.bbc.co.uk/news/60739336?at_medium=...,"Wealthy Russians sanctioned by the US, EU and ..."
4,4,3712,Platinum Jubilee: 70 years of the Queen in 70 ...,"Wed, 01 Jun 2022 23:17:33 GMT",https://www.bbc.co.uk/news/uk-61660128,https://www.bbc.co.uk/news/uk-61660128?at_medi...,A quick look back at the Queen's 70 years on t...


In [5]:
titles=pd.DataFrame(bbc_data["title"])

In [6]:
titles.head()

,title
0,Can I refuse to work?
1,'Liz Truss the Brief?' World reacts to UK poli...
2,Rationing energy is nothing new for off-grid c...
3,The hunt for superyachts of sanctioned Russian...
4,Platinum Jubilee: 70 years of the Queen in 70 ...


In [7]:
titles["lowercase"]=titles["title"].str.lower()

In [8]:
en_stopwords=stopwords.words("english")
titles["lowercase_no_stopwords"]=titles["lowercase"].apply(lambda x :" ".join([ word for word in x.split() if word not in en_stopwords]))

In [9]:
pattern=r"[^\w\s]"
titles["no_punct"]=titles["lowercase_no_stopwords"].apply(lambda x: re.sub(pattern," ",x))

In [10]:
titles["tokens_raw"] = titles["title"].apply(word_tokenize)
titles["tokenized"] = titles["no_punct"].apply(word_tokenize)

In [11]:
lemmatizer=WordNetLemmatizer()
titles["lemmatized"]=titles["tokenized"].apply(lambda x: [lemmatizer.lemmatize(word) for word in x])

In [12]:
tokens_raw_list=sum(titles["lemmatized"],[])
tokens_clean_list=sum(titles["lemmatized"],[])

POS TAGGING

In [13]:
nlp=spacy.load("en_core_web_sm")

In [14]:
spacy_doc=nlp( " ".join(tokens_raw_list))


In [15]:
pos_df=pd.DataFrame(columns=["tokens","pos_tag"])

In [16]:
for token in spacy_doc:
  pos_df=pd.concat([pos_df,pd.DataFrame.from_records([{"tokens":token.text,"pos_tag":token.pos_}])],ignore_index=True)

In [17]:
pos_df.head()

,tokens,pos_tag
0,refuse,AUX
1,work,NOUN
2,liz,PROPN
3,truss,ADJ
4,brief,ADJ


In [18]:
pos_df_counts=pos_df.groupby(["tokens","pos_tag"]).size().reset_index(name="counts").sort_values(by="counts",ascending=False)

In [19]:
nouns=pos_df_counts[pos_df_counts["pos_tag"]=='NOUN']
nouns[:10].head()

,tokens,pos_tag,counts
3793,war,NOUN,36
3897,world,NOUN,32
3886,woman,NOUN,28
3921,year,NOUN,26
900,day,NOUN,22


NER TAGGING

In [20]:
ner_df=pd.DataFrame(columns=["tokens","ner_tag"])

In [22]:
for ent in spacy_doc.ents:
  ner_df=pd.concat([ner_df,pd.DataFrame.from_records([{"tokens":ent.text,"ner_tag":ent.label_}])],ignore_index=True)

In [24]:
ner_df_counts=ner_df.groupby(["tokens","ner_tag"]).size().reset_index(name="counts").sort_values(by="counts",ascending=False)

In [25]:
ner_df_counts.head()

,tokens,ner_tag,counts
40,2022,CARDINAL,26
472,russian,NORP,25
242,first,ORDINAL,17
41,2022,DATE,15
471,russia,GPE,10
